In [57]:
import json 
import torch 
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import evaluate

In [2]:
mt5_finetuned = AutoModelForSeq2SeqLM.from_pretrained("../model/fine-tuned-models/mt5-base-run3")
mt5_tokenizer = AutoTokenizer.from_pretrained("../model/fine-tuned-models/mt5-base-run3")

2025-09-08 14:40:44.939158: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [25]:
tokenizer = AutoTokenizer.from_pretrained("../../mt5-base")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/home/gjergj/.local/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [ ]:
mt5_finetuned.eval()

In [4]:
with open("../INPUT/test_ocr_inputs_mt5-base.json") as f:
    test_set = json.load(f)

In [13]:
keys = list(test_set.keys())
print(type(keys[0]))

<class 'str'>


In [15]:
for k in keys:
    print(test_set[k])
    break

I. Como andò che Maestro Ciliegia, Megnamc trovò un p


As a first step, we need to tokenize and pad each sample to have the same length (20 tokens).

In [18]:
def pad(ocr_chunk):
    if ocr_chunk[-1].item() != 1:       # if needed, add </s> (EOS) token
        ocr_chunk = torch.cat((ocr_chunk, torch.tensor([1])))

    if len(ocr_chunk) < 20:             # if needed, pad to the desired length
        while len(ocr_chunk) < 20:
            ocr_chunk = torch.cat((ocr_chunk, torch.tensor([0])))   

    return ocr_chunk

In [51]:
def chunk_dataset(dataset, tokenizer):
    ''' 
    - dataset: test dataset
    - tokenizer: mt5 tokenizer
    '''

    chunked_ocr_dataset = []

    keys = dataset.keys()

    for k in keys:
        ocr_sample = dataset[k]

        # Compute the token ids
        encoded_ocr = tokenizer.encode(ocr_sample, return_tensors='pt')[0]
        
        ocr_chunk = pad(encoded_ocr)
        # Noticed that sometimes the tokenizer adds a special token '_' at the beginning increasing  
        # the size to 21, we remove it since it was not present in the original file
        if ocr_chunk.shape != torch.Size([20]):
            ocr_chunk = ocr_chunk[1:]

        chunked_ocr_dataset.append(ocr_chunk)

    return chunked_ocr_dataset  
    
            

In [40]:
print(tokenizer.decode([259]))
print(tokenizer.convert_ids_to_tokens([259]))


['▁']


In [52]:
tokenized_test_set = chunk_dataset(test_set, mt5_tokenizer)

In [54]:
for chunk in tokenized_test_set:
    assert chunk.shape == torch.Size([20])
        
print("All fine")

All fine


#### Inference

In [55]:
def postprocess_text(preds):
    preds = [pred.strip() for pred in preds]

    return preds

def clean_text(dataset, model, tokenizer, batch_size):
    predictions = {}
    idx = 0
    num_samples = len(dataset)

    for i in range(0, num_samples, batch_size):
        chunks = dataset[i: i+batch_size]
        batch = torch.stack(chunks, dim=0)

        with torch.no_grad():
            preds = model.generate(batch, num_beams=8, num_return_sequences=1)
            
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

        postprocessed_preds = postprocess_text(decoded_preds)

        for pred in postprocessed_preds:
            predictions[idx] = pred
            idx += 1 

    return predictions



In [56]:
cleaned_text = clean_text(tokenized_test_set, mt5_finetuned, mt5_tokenizer, batch_size=8)

In [ ]:
# Save the results
with open("../OUTPUT/AILovePython-hw2_ocr-mt5-base.json", "w", encoding='utf-8') as f:
    json.dump(cleaned_text, f, ensure_ascii=False, indent=4)

#### Rouge scores

In [58]:
metric = evaluate.load("rouge")

Using the latest cached version of the module from /home/gjergj/.cache/huggingface/modules/evaluate_modules/metrics/evaluate-metric--rouge/b01e0accf3bd6dd24839b769a5fda24e14995071570870922c71970b3a6ed886 (last modified on Thu Aug 21 19:39:02 2025) since it couldn't be found locally at evaluate-metric--rouge, or remotely on the Hugging Face Hub.


In [63]:
with open("../INPUT/test_clean_samples_mt5-base.json") as f:
    references = json.load(f)

In [67]:
with open("../OUTPUT/AILovePython-hw2_ocr-mt5-base.json") as f:
    predictions = json.load(f)

In [76]:
preds = []
refs = []

keys = list(references.keys())

for k in keys:
    preds.append(predictions[k])
    refs.append(references[k])

rouge_scores = metric.compute(predictions=preds, references=refs)
print(rouge_scores)

{'rouge1': 0.932846165176281, 'rouge2': 0.8619417992654992, 'rougeL': 0.9330499461410855, 'rougeLsum': 0.9331806583650661}
